# 01 – Data preparation and exploratory profile
**QM640 Data Analytics Capstone – Binesh Balakrishnan**

Predicting SLA breach in IT incident management. **Dependent variable for all four research questions: SLA breach** (1 = breached, 0 = met).

Source: Amaral, C., Fantinato, M., & Peres, S. (2018). *Incident management process enriched event log* [Data set]. UCI Machine Learning Repository. https://doi.org/10.24432/C57S4H (CC BY 4.0).

This notebook:
1. loads the raw event log (141,712 events, 36 columns);
2. collapses it to one row per incident (first record = at-logging attributes; final record = outcome and accumulated counts);
3. derives `sla_breach` (the outcome), `reassigned`, and `resolution_hours` (descriptive only);
4. confirms the polarity of `made_sla`;
5. writes `data/processed/incidents.csv` and reproduces the descriptive profile (synopsis Table 6);
6. computes the data-derived inputs used in the sample size calculation (synopsis Tables 2–3).

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

RAW = Path('../data/raw/incident_event_log.csv')
OUT = Path('../data/processed'); OUT.mkdir(parents=True, exist_ok=True)
DATE_COLS = ['opened_at', 'sys_created_at', 'sys_updated_at', 'resolved_at', 'closed_at']

## 1. Load and validate

In [2]:
ev = pd.read_csv(RAW, na_values='?', keep_default_na=False, low_memory=False)
for c in DATE_COLS:
    ev[c] = pd.to_datetime(ev[c], format='%d/%m/%Y %H:%M', errors='coerce')

assert ev.shape == (141712, 36), ev.shape
assert ev['number'].nunique() == 24918
print(ev.shape, ev['number'].nunique(), 'incidents')
ev.isna().mean().sort_values(ascending=False).round(3).head(12)

(141712, 36) 24918 incidents


caused_by           1.000
vendor              0.998
cmdb_ci             0.997
rfc                 0.993
problem_id          0.984
sys_created_by      0.375
sys_created_at      0.375
u_symptom           0.233
assigned_to         0.194
assignment_group    0.100
opened_by           0.034
resolved_at         0.022
dtype: float64

In [3]:
ev['incident_state'].value_counts()

incident_state
Active                38716
New                   36407
Resolved              25751
Closed                24985
Awaiting User Info    14642
Awaiting Vendor         707
Awaiting Problem        461
Awaiting Evidence        38
-100                      5
Name: count, dtype: int64

## 2. Collapse to one row per incident

In [4]:
ev = ev.sort_values(['number', 'sys_mod_count', 'sys_updated_at'])
first = ev.groupby('number').head(1).set_index('number')
last = ev.groupby('number').tail(1).set_index('number')
assert (last['incident_state'] == 'Closed').all()
print(len(first), len(last))

24918 24918


## 3. Derive analysis variables

In [5]:
inc = pd.DataFrame(index=last.index)
# outcome and accumulated counts (final record) - RQ1 to RQ3
for c in ['priority', 'impact', 'urgency', 'category', 'subcategory', 'knowledge',
          'reassignment_count', 'reopen_count', 'sys_mod_count', 'made_sla',
          'contact_type', 'opened_at', 'resolved_at', 'closed_at']:
    inc[c] = last[c]
# resolution time is descriptive only (not an outcome)
inc['resolution_hours'] = (last['resolved_at'] - last['opened_at']).dt.total_seconds() / 3600
inc['sla_breach'] = (~last['made_sla']).astype(int)
inc['reassigned'] = (last['reassignment_count'] >= 1).astype(int)
# at-logging attributes (first record) - RQ4 predictors
for c in ['priority', 'impact', 'urgency', 'category', 'subcategory', 'u_symptom', 'contact_type',
          'location', 'assignment_group', 'caller_id', 'u_priority_confirmation', 'notify', 'cmdb_ci', 'knowledge']:
    inc[f'log_{c}'] = first[c]
inc['log_hour'] = inc['opened_at'].dt.hour
inc['log_dayofweek'] = inc['opened_at'].dt.dayofweek
inc['valid_resolution'] = inc['resolution_hours'] > 0
inc.to_csv(OUT / 'incidents.csv')
inc.shape

(24918, 34)

## 4. Confirm `made_sla` polarity

In [6]:
# true should mean the SLA was met: resolution time should be far shorter
inc[inc.valid_resolution].groupby('made_sla')['resolution_hours'].median().round(1)

made_sla
False    201.7
True       1.1
Name: resolution_hours, dtype: float64

## 5. Outcome coverage and descriptive profile (synopsis Table 6)

In [7]:
print('Incidents (all RQs)       :', len(inc))
print('SLA outcome missing       :', inc['sla_breach'].isna().sum())
print('Breaches                  :', int(inc['sla_breach'].sum()))
print('Breach prevalence         :', round(inc['sla_breach'].mean(), 3))
print('Resolution time missing   :', inc['resolution_hours'].isna().sum(), '(descriptive only)')
print('Resolution time zero      :', (inc['resolution_hours'] == 0).sum(), '(descriptive only)')

Incidents (all RQs)       : 24918
SLA outcome missing       : 0
Breaches                  : 9115
Breach prevalence         : 0.366
Resolution time missing   : 1556 (descriptive only)
Resolution time zero      : 39 (descriptive only)


In [8]:
profile = pd.DataFrame({
    'incidents': inc.groupby('priority').size(),
    'share_%': (inc['priority'].value_counts(normalize=True) * 100).round(1),
    'breach_rate_%': (inc.groupby('priority')['sla_breach'].mean() * 100).round(1),
    'reassigned_%': (inc.groupby('priority')['reassigned'].mean() * 100).round(1),
    'knowledge_used_%': (inc.groupby('priority')['knowledge'].mean() * 100).round(1),
})
profile.loc['All'] = [len(inc), 100.0, round(inc['sla_breach'].mean() * 100, 1),
                      round(inc['reassigned'].mean() * 100, 1), round(inc['knowledge'].mean() * 100, 1)]
profile

,incidents,share_%,breach_rate_%,reassigned_%,knowledge_used_%
priority,,,,,
1 - Critical,270.0,1.1,98.1,58.5,12.2
2 - High,408.0,1.6,99.5,62.5,19.4
3 - Moderate,23466.0,94.2,35.5,45.4,13.7
4 - Low,774.0,3.1,15.9,37.9,30.2
All,24918.0,100.0,36.6,45.6,14.3


In [9]:
print('Breach rate by reassigned (0/1) :', (inc.groupby('reassigned')['sla_breach'].mean() * 100).round(1).to_dict())
print('Breach rate by knowledge (F/T)  :', (inc.groupby('knowledge')['sla_breach'].mean() * 100).round(1).to_dict())
print('Knowledge used (F:T)            :', inc['knowledge'].value_counts().to_dict())
print('Contact channel share           :', (inc['contact_type'].value_counts(normalize=True) * 100).round(1).to_dict())
print('Opened by month:'); inc['opened_at'].dt.to_period('M').value_counts().sort_index()

Breach rate by reassigned (0/1) : {0: 21.6, 1: 54.4}
Breach rate by knowledge (F/T)  : {False: 30.5, True: 73.2}
Knowledge used (F:T)            : {False: 21367, True: 3551}
Contact channel share           : {'Phone': 99.1, 'Self service': 0.6, 'Email': 0.2, 'IVR': 0.0, 'Direct opening': 0.0}
Opened by month:


opened_at
2016-02     207
2016-03    8995
2016-04    7934
2016-05    7508
2016-06       5
2016-07      14
2016-08      15
2016-09      12
2016-10      16
2016-11      26
2016-12      37
2017-01      94
2017-02      55
Freq: M, Name: count, dtype: int64

## 6. Inputs for the sample size calculation (synopsis Tables 2–3)

R² of each independent variable on the control variables (priority and category), used to inflate the χ² sample size for RQ2 and RQ3; the smallest expected cell count for RQ1; and the breach prevalence for RQ4.

In [10]:
from scipy.stats import chi2_contingency
X = pd.get_dummies(inc[['priority', 'category']].fillna('NA'), drop_first=True).astype(float)
X.insert(0, 'const', 1.0)
def r2(y):
    y = y.astype(float).values
    b, *_ = np.linalg.lstsq(X.values, y, rcond=None)
    return 1 - ((y - X.values @ b) ** 2).sum() / ((y - y.mean()) ** 2).sum()
print('R² reassignment_count ~ priority + category (RQ2):', round(r2(inc['reassignment_count']), 3))
print('R² knowledge ~ priority + category (RQ3)         :', round(r2(inc['knowledge']), 3))
expected = chi2_contingency(pd.crosstab(inc['priority'], inc['sla_breach']))[3]
print('Smallest expected cell count, priority x breach (RQ1):', round(expected.min(), 1))
print('Breach prevalence (RQ4):', round(inc['sla_breach'].mean(), 3))

R² reassignment_count ~ priority + category (RQ2): 0.047
R² knowledge ~ priority + category (RQ3)         : 0.154
Smallest expected cell count, priority x breach (RQ1): 98.8
Breach prevalence (RQ4): 0.366
